In [1]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 05, Cartan calculus

Companion notebook to [05_cartan_calculus.md](05_cartan_calculus.md). The five Cartan relations as `OperatorEquation`s, the axiom/theorem split for `d² = 0`, live verification of the magic formula in both modes, and the invariant-d helper.

## The bundle

`CartanCalculus(d, L, ι, [·,·])`, four ingredients in one object.

In [2]:
from jacopy.algebra.derivation import Derivation
from jacopy.brackets.lie import LieBracket
from jacopy.calculus.cartan import CartanCalculus, RELATIONS
from jacopy.calculus.exterior_algebra import ExteriorAlgebra
from jacopy.calculus.exterior_d import d
from jacopy.calculus.interior import interior
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry

cart = CartanCalculus(
    d=d, lie_derivative=lie_derivative,
    interior=interior, vector_bracket=LieBracket(),
)
print('RELATIONS:', RELATIONS)

RELATIONS: ('d_squared_zero', 'cartan_magic', 'd_lie', 'lie_lie', 'lie_iota')


## Five relations, five `OperatorEquation`s

In [3]:
reg = PropertyRegistry()
f = Symbol("f")
reg.declare(f, Graded(degree=0))
algebra = ExteriorAlgebra((f,))
X = Derivation("X", degree=0)
Y = Derivation("Y", degree=0)

for name, kw in [
    ("d_squared_zero", {}),
    ("cartan_magic", {"X": X}),
    ("d_lie", {"X": X}),
    ("lie_lie", {"X": X, "Y": Y}),
    ("lie_iota", {"X": X, "Y": Y}),
]:
    eq = cart.relation(name, algebra=algebra, **kw)
    print(f"{name:16s}: {eq.lhs} = {eq.rhs}")

d_squared_zero  : (d * d) = 0
cartan_magic    : ((d * ι_X) + (ι_X * d)) = L_X
d_lie           : ((d * L_X) + (-(L_X * d))) = 0
lie_lie         : ((L_X * L_Y) + (-(L_Y * L_X))) = L_((X * Y) + (-(Y * X)))
lie_iota        : ((L_X * ι_Y) + (-(ι_Y * L_X))) = ι_((X * Y) + (-(Y * X)))


## `d² = 0`, axiom mode vs theorem mode

The plain helper `apply_d_squared_zero` always rewrites to `0`. The default engine with `d_squared_mode="theorem"` and foundational mode records `d(d(x)) → 0` as a `ProofStep`.

In [4]:
from jacopy.calculus.exterior_d import apply_d_squared_zero
from jacopy.proof.expansion import default_engine

x = Symbol("x")
reg.declare(x, Graded(degree=0))
print('axiom rewrite:', apply_d_squared_zero(d(d(x))))

engine = default_engine(
    registry=reg, mode="foundational", d_squared_mode="theorem"
)
expanded, steps = engine.expand(d(d(x)))
print('theorem-mode expanded:', expanded)
print('theorem-mode step rule:', steps[0].rule)

axiom rewrite: 0
theorem-mode expanded: 0
theorem-mode step rule: d² = 0


## All five relations, live proof via `verify`

`cartan_magic` closes in a single step in both modes on `ExteriorAlgebra((f,))`; the other four close at the generator level via `AgreementOnGenerators` + `ExpandAndSimplify`.

In [5]:
chain = cart.verify("cartan_magic", algebra=algebra, X=X, registry=reg)
print('efficient len:', len(chain), 'rule:', chain.steps[0].rule)

chain_f = cart.verify(
    "cartan_magic", algebra=algebra, X=X, registry=reg,
    mode="foundational",
)
print('foundational len:', len(chain_f), 'rule:', chain_f.steps[0].rule)

results = cart.verify_all(algebra=algebra, X=X, Y=Y, registry=reg)
for name, c in results.items():
    print(f'{name:16s}: len={len(c)}')

efficient len: 1 rule: AgreementOnGenerators
foundational len: 1 rule: AgreementOnGenerators
d_squared_zero  : len=1
cartan_magic    : len=1
d_lie           : len=1
lie_lie         : len=1
lie_iota        : len=1


## `invariant_d`, theorem derived from magic + lie_iota

`dω(X, Y) = X(ω(Y)) − Y(ω(X)) − ω([X, Y])`, the Koszul-Cartan invariant formula for 1-forms. `InvariantDOneFormDefinition`'s default classification is `"theorem"` (in contrast to `d²=0`'s `"axiom"`), the formula falls out naturally from magic + lie_iota.

In [6]:
from jacopy.calculus.invariant_d import invariant_d_one_form
from jacopy.brackets.lie import lie

omega = Symbol("ω")
reg.declare(omega, Graded(degree=1))
print(invariant_d_one_form(omega, X, Y, bracket=lie))

(X(ι_Y(ω)) + (-Y(ι_X(ω))) + (-ι_((X * Y) + (-(Y * X)))(ω)))


## Twisted Cartan bundle, `d_H = d + H∧`

For a closed 3-form `H`, the H-twisted exterior derivative `d_H` carries the Cartan calculus through the same five relations. The `TwistedCartanBundle(H)` wrapper builds `d_H` as a fresh `ExteriorDerivative` and rewires the Lie-derivative factory so `d_H` slots into the bundle. Constructing the bundle is making the assumption `dH = 0`.

In [7]:
from jacopy.library import TwistedCartanBundle

H = Symbol("H")
reg.declare(H, Graded(degree=3))
bundle = TwistedCartanBundle(H)
print('bundle.d:', bundle.d)
print('bundle.cartan.d:', bundle.cartan.d)

algebra_H = ExteriorAlgebra((f,), d=bundle.d)
results = bundle.cartan.verify_all(
    algebra=algebra_H, X=X, Y=Y, registry=reg
)
for name, c in results.items():
    print(f'{name:16s}: len={len(c)}')

bundle.d: d_H
bundle.cartan.d: d_H
d_squared_zero  : len=1
cartan_magic    : len=1
d_lie           : len=1
lie_lie         : len=1
lie_iota        : len=1


## Next step

Building your own bracket + the Jacobi test, [06_custom_bracket.md](06_custom_bracket.md) (Stage C).